In [6]:
import pandas as pd
import numpy as np
from zero_point import zpt
zpt.load_tables()

In [7]:
# Funciones vectorizadas para correccion de paralaje y movimiento propio
# Usando operaciones de numpy para máximo rendimiento

def get_rotation_vectorized(G):
    """
    Calcula wx, wy, wz de forma vectorizada para arrays de magnitudes.
    
    Parameters
    ----------
    G : np.ndarray
        Array de magnitudes G
        
    Returns
    -------
    tuple of np.ndarray
        (wx, wy, wz) arrays
    """
    wx = np.full_like(G, 0, dtype=np.float64)
    wy = np.full_like(G, 0, dtype=np.float64)
    wz = np.full_like(G, 0, dtype=np.float64)
    
    mask1 = G < 9
    wx[mask1], wy[mask1], wz[mask1] = -5, -3, 0
    
    mask2 = (G >= 9) & (G < 11)
    wx[mask2], wy[mask2], wz[mask2] = -10, -5, 2
    
    mask3 = (G >= 11) & (G < 13)
    wx[mask3], wy[mask3], wz[mask3] = -25, -15, 5
    
    return wx, wy, wz

def pm_correction_vectorized(mu_ra, mu_dec, ra, dec, wx, wy, wz):
    """
    Corrige movimiento propio de forma vectorizada usando operaciones numpy.
    
    Parameters
    ----------
    mu_ra, mu_dec : np.ndarray
        Movimientos propios en RA y Dec (mas/yr)
    ra, dec : np.ndarray
        Coordenadas ecuatoriales (grados)
    wx, wy, wz : np.ndarray
        Parámetros de rotación del sistema de referencia
        
    Returns
    -------
    tuple of np.ndarray
        (mu_ra_corr, mu_dec_corr) corregidos
    """
    # Conversión a radianes (vectorizado)
    ra_rad = np.deg2rad(ra)
    dec_rad = np.deg2rad(dec)
    
    # Pre-computar funciones trigonométricas
    sin_ra = np.sin(ra_rad)
    cos_ra = np.cos(ra_rad)
    sin_dec = np.sin(dec_rad)
    cos_dec = np.cos(dec_rad)
    
    # Correcciones (operaciones vectorizadas)
    dmu_ra = -wx * sin_ra + wy * cos_ra
    
    dmu_dec = (-wx * cos_ra * sin_dec 
               - wy * sin_ra * sin_dec 
               + wz * cos_dec)
    
    # Aplicar correcciones
    mu_ra_corr = mu_ra - dmu_ra
    mu_dec_corr = mu_dec - dmu_dec
    
    return mu_ra_corr, mu_dec_corr


In [8]:
df_gaia = pd.read_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\DatosTotales\gaia_parallax5f.csv", index_col=0)
df_sp = pd.read_csv("C:\\Users\\nicob\\One Drive Uniandes\\OneDrive - Universidad de los Andes\\Doctorado\\proyecto\\clusterization_project\\data\\datos_shell\\fidelity_table.csv")

In [9]:
# aplicando correccion de paralaje
df_gaia["zp"] = zpt.get_zpt(
    df_gaia["phot_g_mean_mag"],
    df_gaia["nu_eff_used_in_astrometry"],
    df_gaia["pseudocolour"],
    df_gaia["ecl_lat"],
    df_gaia["astrometric_params_solved"]
)

c:\Users\nicob\anaconda3\Lib\site-packages\zero_point\zpt.py:215: UserWarning: The apparent magnitude of one or more of the sources is outside the expected range (6-21 mag). 
                Outside this range, there is no further interpolation, thus the values at 6 or 21 are returned.
  warnings.warn(
c:\Users\nicob\anaconda3\Lib\site-packages\zero_point\zpt.py:230: UserWarning: The nu_eff_used_in_astrometry of some of the 5p source(s) is outside the expected range (1.1-1.9 
                mag). Outside this range, the zero-point calculated can be seriously wrong.
  warnings.warn(
c:\Users\nicob\anaconda3\Lib\site-packages\zero_point\zpt.py:243: UserWarning: The pseudocolour of some of the 6p source(s) is outside the expected range (1.24-1.72 mag).
                 The maximum corrections are reached already at 1.24 and 1.72
  warnings.warn(


In [10]:
df_gaia["parallax_corrected"] = df_gaia["parallax"] - df_gaia["zp"]/1000.0

In [11]:
df = df_gaia.merge(df_sp, on='source_id', how='inner')

In [12]:
# Aplicar correcciones de rotacion y movimiento propio en una sola operacion vectorizada
# Esto es O(n) en lugar de O(n*m) con apply()

# Obtener arrays numpy para operaciones vectorizadas
G = df['phot_g_mean_mag'].values
ra = df['ra'].values
dec = df['dec'].values
mu_ra = df['pmra'].values
mu_dec = df['pmdec'].values

# Calcular parámetros de rotación
wx, wy, wz = get_rotation_vectorized(G)

# Aplicar correcciones de movimiento propio
mu_ra_corr, mu_dec_corr = pm_correction_vectorized(mu_ra, mu_dec, ra, dec, wx, wy, wz)

# Asignar resultados al dataframe
df['wx'] = wx
df['wy'] = wy
df['wz'] = wz
df['pmra_corrected'] = mu_ra_corr
df['pmdec_corrected'] = mu_dec_corr


In [13]:
# Validación y estadísticas de las correcciones aplicadas
print(f"Total de estrellas procesadas: {len(df):,}")
print(f"\nRangos de parámetros de rotación:")
print(f"  wx: [{df['wx'].min():.1f}, {df['wx'].max():.1f}] (media: {df['wx'].mean():.2f})")
print(f"  wy: [{df['wy'].min():.1f}, {df['wy'].max():.1f}] (media: {df['wy'].mean():.2f})")
print(f"  wz: [{df['wz'].min():.1f}, {df['wz'].max():.1f}] (media: {df['wz'].mean():.2f})")

print(f"\nCorrecciones de movimiento propio (mas/yr):")
print(f"  pmra_delta: media = {(df['pmra_corrected'] - df['pmra']).mean():.4f}, "
      f"std = {(df['pmra_corrected'] - df['pmra']).std():.4f}")
print(f"  pmdec_delta: media = {(df['pmdec_corrected'] - df['pmdec']).mean():.4f}, "
      f"std = {(df['pmdec_corrected'] - df['pmdec']).std():.4f}")

print(f"\nCorrelación entre componentes de movimiento propio original:")
print(f"  r(pmra, pmdec) = {np.corrcoef(df['pmra'], df['pmdec'])[0,1]:.4f}")
print(f"\nCorrelación entre componentes de movimiento propio corregidas:")
print(f"  r(pmra_corr, pmdec_corr) = {np.corrcoef(df['pmra_corrected'], df['pmdec_corrected'])[0,1]:.4f}")


Total de estrellas procesadas: 574,531

Rangos de parámetros de rotación:
  wx: [-25.0, 0.0] (media: -2.21)
  wy: [-15.0, 0.0] (media: -1.28)
  wz: [0.0, 5.0] (media: 0.41)

Correcciones de movimiento propio (mas/yr):
  pmra_delta: media = 0.0301, std = 5.5466
  pmdec_delta: media = -0.3493, std = 3.3817

Correlación entre componentes de movimiento propio original:
  r(pmra, pmdec) = -0.0242

Correlación entre componentes de movimiento propio corregidas:
  r(pmra_corr, pmdec_corr) = -0.0245


In [14]:
# Benchmark: Comparación de eficiencia con la solución anterior (solo para referencia)
import time

# Crear dataframe de prueba con subset de datos para benchmark
sample_size = min(10000, len(df))
df_sample = df.iloc[:sample_size].copy()

# Método anterior (ineficiente con apply + lambda)
def benchmark_apply_method():
    def get_rotation(G):
        if G < 9:
            return -5, -3, 0
        elif G < 11:
            return -10, -5, 2
        elif G < 13:
            return -25, -15, 5
        else:
            return 0, 0, 0
    
    df_test = df_sample.copy()
    start = time.perf_counter()
    df_test[['wx', 'wy', 'wz']] = df_test['phot_g_mean_mag'].apply(lambda G: pd.Series(get_rotation(G)))
    t1 = time.perf_counter() - start
    return t1

# Método actual (vectorizado)
def benchmark_vectorized_method():
    start = time.perf_counter()
    G = df_sample['phot_g_mean_mag'].values
    wx, wy, wz = get_rotation_vectorized(G)
    t2 = time.perf_counter() - start
    return t2

t_apply = benchmark_apply_method()
t_vect = benchmark_vectorized_method()

print(f"Benchmark ({sample_size} filas):")
print(f"  Método con apply():     {t_apply*1000:.2f} ms")
print(f"  Método vectorizado:     {t_vect*1000:.2f} ms")
print(f"  Speedup:                {t_apply/t_vect:.1f}x más rápido")
print(f"\nEn dataset completo ({len(df):,} filas):")
print(f"  Estimado (vectorizado): {(t_vect * len(df) / sample_size)*1000:.0f} ms")


Benchmark (10000 filas):
  Método con apply():     1276.80 ms
  Método vectorizado:     0.36 ms
  Speedup:                3576.5x más rápido

En dataset completo (574,531 filas):
  Estimado (vectorizado): 21 ms


In [15]:
print(len(df),len(df[df['fidelity_v2']>0.5]))

574531 300447


In [17]:
df.reset_index(drop=True).to_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\data\datos_shell\gaia_parallax5_10_fidelity.csv")

In [16]:
df

,source_id,ra,dec,parallax,pmra,pmdec,ruwe,phot_g_mean_mag,bp_rp,radial_velocity,...,astrometric_params_solved,zp,parallax_corrected,fidelity_v2,fidelity_v1,wx,wy,wz,pmra_corrected,pmdec_corrected
0,83154862613888,44.937316,0.623793,15.672104,-98.230681,-160.625162,1.056748,9.542494,0.988867,-37.740585,...,31,-0.031544,15.672136,1.0,1.0,-10.0,-5.0,2.0,-101.754609,-162.740560
1,1381403216912512,44.478163,2.588188,11.370985,-21.507905,18.684881,5.531934,12.760104,1.926095,8.692732,...,31,-0.014312,11.370999,1.0,1.0,-25.0,-15.0,5.0,-28.321076,12.409893
2,1593815119566464,44.108751,2.928773,12.804964,230.166109,100.853354,1.205072,14.549130,2.493920,13.428328,...,31,-0.051234,12.805015,1.0,1.0,0.0,0.0,0.0,230.166109,100.853354
3,1651676918907264,43.252673,3.175368,14.773574,54.553329,-95.068482,1.122411,14.482400,2.519452,1.442179,...,31,-0.051439,14.773626,1.0,1.0,0.0,0.0,0.0,54.553329,-95.068482
4,2725311368387968,48.419232,4.256734,10.086866,-73.955081,-174.992089,1.062597,16.194202,2.677515,NaN,...,31,-0.045808,10.086911,1.0,1.0,0.0,0.0,0.0,-73.955081,-174.992089
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
574526,6916406185752398976,313.712816,-3.439946,10.991049,29.798323,9.020685,0.980640,18.533690,0.511457,NaN,...,31,-0.019354,10.991069,1.0,1.0,0.0,0.0,0.0,29.798323,9.020685
574527,6916692368012771200,312.829062,-2.778627,11.656269,55.594115,9.984257,1.906532,10.886308,1.218915,-0.407206,...,31,-0.034535,11.656304,1.0,1.0,-10.0,-5.0,2.0,66.327033,8.138400
574528,6917223054171248256,315.903394,-0.929389,12.918603,149.959540,-44.636416,0.932791,19.328259,1.295691,NaN,...,95,-0.012780,12.918616,1.0,1.0,0.0,0.0,0.0,149.959540,-44.636416
574529,6917290163036289408,314.673110,-1.273830,27.520591,73.157189,-17.405506,1.076358,16.673510,4.079328,NaN,...,95,-0.063126,27.520655,1.0,1.0,0.0,0.0,0.0,73.157189,-17.405506
